# UCS420 Lab Assignment 9 — NLP Text Preprocessing
Enter your roll number and name, then run all cells.

In [15]:
ROLL_NO = 1024170012        # <-- your full roll number
NAME = "Riya"            # <-- your name

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"

### Dataset generator — do not modify

In [16]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline — do not modify

In [17]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [18]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

Roll number: 1024170012   Name: Riya   Fingerprint: 96BE8EA4   Tickets: 14

Ticket_ID                                                                                  Ticket_Text
      T01                             Projector keeps disconnecting during the lab. Please resolve it.
      T02                       VPN keeps asking for login during the lab. I have tried several times!
      T03           I couldn't connect to the student portal after resetting my password. Please help.
      T04                                  Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it.
      T05                                The projector has stopped syncing. Is there another solution?
      T06                                                        Nothing works and nobody has replied.
      T07                                                         Python 3.12 was installed yesterday.
      T08                                                  The state-of-the-art lab (Lab-3) is close

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [19]:
# Your code
# Run the official preprocessing function on every ticket.
processed = {
    row.Ticket_ID: process_ticket(row.Ticket_Text)
    for row in df.itertuples(index=False)
}

rows = []

for ticket_id, result in processed.items():
    rows.append({
        "Ticket_ID": ticket_id,
        "Sentences": len(result["sentences"]),
        "Tokens": len(result["tokens"]),
        "Punctuation Tokens": len(result["punct"]),
        "Stopwords": len(result["stops"]),
        "Unique Tokens": len(set(result["tokens"])),
        "Final Tokens": len(result["final"])
    })

q1_table = pd.DataFrame(rows)

# TOTAL row: unique tokens means unique across the entire corpus.
total_row = {
    "Ticket_ID": "TOTAL",
    "Sentences": q1_table["Sentences"].sum(),
    "Tokens": q1_table["Tokens"].sum(),
    "Punctuation Tokens": q1_table["Punctuation Tokens"].sum(),
    "Stopwords": q1_table["Stopwords"].sum(),
    "Unique Tokens": len(set(
        token
        for result in processed.values()
        for token in result["tokens"]
    )),
    "Final Tokens": q1_table["Final Tokens"].sum()
}

q1_table = pd.concat(
    [q1_table, pd.DataFrame([total_row])],
    ignore_index=True
)

print(q1_table.to_string(index=False))

Ticket_ID  Sentences  Tokens  Punctuation Tokens  Stopwords  Unique Tokens  Final Tokens
      T01          2      11                   2          3             10             6
      T02          2      15                   2          5             15             8
      T03          2      16                   2          5             15             9
      T04          1       9                   2          2              9             5
      T05          2      11                   2          4             11             5
      T06          1       7                   1          2              7             4
      T07          1       6                   1          1              6             4
      T08          1       9                   3          2              9             4
      T09          2      18                   2          9             18             7
      T10          2      14                   2          4             12             8
      T11          2 

**Observe:** Which final tokens in your data are not real words (e.g. n't, 's, ca)? Where do they come from?

*Your answer: Tokens such as n't, 's, and ca can arise because NLTK's word_tokenize() splits contractions into multiple tokens. Since the official pipeline removes punctuation and stopwords but does not rejoin contractions, some fragments remain in the final tokens.*

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [20]:
TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

# Your code
import re

# Combine raw tickets into one string.
raw_tickets = df["Ticket_Text"].tolist()
all_text = "\n".join(raw_tickets)

# 1. Words longer than 5 letters.
long_words = re.findall(
    r"\b[A-Za-z]{6,}\b",
    all_text
)

# 2. Integers, decimals, versions and comma-separated numbers.
number_pattern = (
    r"(?<![\w.])"
    r"(?:\d{1,3}(?:,\d{3})+|\d+)"
    r"(?:\.\d+)*(?![\w.])"
)

numbers = re.findall(number_pattern, all_text)

# 3. Capitalized words.
capitalized_words = re.findall(
    r"\b[A-Z][a-zA-Z]*\b",
    all_text
)

# 4. Words beginning with a vowel.
vowel_words = re.findall(
    r"\b[AEIOUaeiou][A-Za-z]*\b",
    all_text
)

print("Words longer than 5 letters:", long_words)
print("\nNumbers:", numbers)
print("\nCapitalized words:", capitalized_words)
print("\nWords starting with a vowel:", vowel_words)

Words longer than 5 letters: ['Projector', 'disconnecting', 'during', 'Please', 'resolve', 'asking', 'during', 'several', 'couldn', 'connect', 'student', 'portal', 'resetting', 'password', 'Please', 'helpdesk', 'thapar', 'thapar', 'projector', 'stopped', 'syncing', 'another', 'solution', 'Nothing', 'nobody', 'replied', 'Python', 'installed', 'yesterday', 'closed', 'locked', 'projector', 'connected', 'campus', 'another', 'solution', 'couldn', 'connect', 'yesterday', 'worked', 'yesterday', 'MATLAB', 'crashes', 'opening', 'another', 'computer', 'crashes', 'opening', 'connected', 'campus', 'another', 'solution', 'stopped', 'responding', 'several', 'projector', 'disconnecting', 'several']

Numbers: ['3.12', '3']

Capitalized words: ['Projector', 'Please', 'VPN', 'I', 'I', 'Please', 'Write', 'The', 'Is', 'Nothing', 'Python', 'The', 'Lab', 'I', 'Wi', 'Fi', 'Is', 'I', 'WiFi', 'It', 'The', 'MATLAB', 'It', 'Email', 'Wi', 'Fi', 'Is', 'The', 'I', 'The', 'I']

Words starting with a vowel: ['it', 'a

In [21]:
EMAIL_RE = (
    r"(?<![\w.+-])"
    r"[A-Za-z0-9_%+-]+(?:\.[A-Za-z0-9_%+-]+)*"
    r"@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+"
    r"(?![\w.-])"
)

URL_RE = r"\b(?:https?://|www\.)[^\s<>\"']+"

PHONE_RE = (
    r"(?<![\w.])"
    r"(?:\+\d{1,3}[-.\s]?)?"
    r"(?:\(\d{2,4}\)[-.\s]?|\d{3,4}[-.\s])?"
    r"\d{3,5}[-.\s]\d{3,5}"
    r"(?![\w.])"
)


def replace_sensitive_info(text):
    # Replace emails first to avoid treating them as URLs.
    text = re.sub(EMAIL_RE, "<EMAIL>", text)

    # Remove trailing sentence punctuation from the matched URL
    # so that punctuation remains outside the placeholder.
    text = re.sub(URL_RE, "<URL>", text)

    text = re.sub(PHONE_RE, "<PHONE>", text)

    return text


for ticket_id, text in zip(df["Ticket_ID"], raw_tickets):
    print(ticket_id, ":", replace_sensitive_info(text))

T01 : Projector keeps disconnecting during the lab. Please resolve it.
T02 : VPN keeps asking for login during the lab. I have tried several times!
T03 : I couldn't connect to the student portal after resetting my password. Please help.
T04 : Write to <EMAIL> or visit <URL>
T05 : The projector has stopped syncing. Is there another solution?
T06 : Nothing works and nobody has replied.
T07 : Python 3.12 was installed yesterday.
T08 : The state-of-the-art lab (Lab-3) is closed.
T09 : I was locked out of the projector when connected to campus Wi-Fi. Is there another solution?
T10 : I couldn't connect to the WiFi since yesterday. It worked yesterday.
T11 : The MATLAB crashes while opening. It works on another computer.
T12 : Email crashes while opening when connected to campus Wi-Fi. Is there another solution?
T13 : The e-mail stopped responding. I have tried several times!
T14 : The projector keeps disconnecting. I have tried several times!


In [22]:
for line in TEST_LINES:
    print("\nOriginal:", line)

    print("Long words:",
          re.findall(r"\b[A-Za-z]{6,}\b", line))

    print("Numbers:",
          re.findall(number_pattern, line))

    print("Capitalized:",
          re.findall(r"\b[A-Z][a-zA-Z]*\b", line))

    print("Vowel words:",
          re.findall(r"\b[AEIOUaeiou][A-Za-z]*\b", line))

    print("Replaced:", replace_sensitive_info(line))


Original: Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.
Long words: ['helpdesk', 'thapar', 'thapar', 'thapar']
Numbers: []
Capitalized: ['Mail']
Vowel words: ['it', 'edu', 'or', 'a', 'univ', 'ac', 'in', 'edu', 'or', 'edu', 'it']
Replaced: Mail <EMAIL> or <EMAIL>; visit <URL> or <URL>

Original: Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.
Long words: ['Version']
Numbers: ['91', '98765', '43210', '91', '98765', '43210', '0175', '239', '2.0.1', '95.5', '1']
Capitalized: ['Call', 'Version', 'Rs']
Vowel words: ['or', 'is']
Replaced: Call <PHONE>, <PHONE> or <PHONE>-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.

Original: The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.
Long words: []
Numbers: ['5.30']
Capitalized: ['The']
Vowel words: ['of', 'art', 'isn', 'open', 'in', 'at']
Replaced: The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't w

**Observe:** How many of your capitalized words are capitalized only because they start a sentence?

*Your answer:Some capitalized words are proper nouns or technical names, while others are capitalized simply because they begin a sentence. Sentence position alone does not establish whether a word is a proper noun.*

---
## Q3. Custom tokenizer
Write my_tokenize(text) with one RegexpTokenizer pattern that keeps contractions (isn't), hyphenated words (Wi-Fi, state-of-the-art), decimals and versions (95.5, 2.0.1), emails, URLs and phone numbers as single tokens. Run it on TEST_LINES and on your tickets, and compare with word_tokenize.

In [23]:
# Your code
from nltk.tokenize import RegexpTokenizer, word_tokenize

TOKEN_PATTERN = r"""
    (?:https?://|www\.)[^\s<>"']+
    |
    [A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}
    |
    \+\d{1,3}[-.\s]\d{3,5}[-.\s]\d{3,5}
    |
    \d{3,5}[-.\s]\d{3,5}[-.\s]\d{3,5}
    |
    \d{3,4}[-.\s]\d{3,5}
    |
    \d+(?:\.\d+){2,}
    |
    \d+\.\d+
    |
    \d{1,3}(?:,\d{3})+
    |
    [A-Za-z]+(?:-[A-Za-z]+)+
    |
    [A-Za-z]+(?:['’][A-Za-z]+)+
    |
    [A-Za-z]+
    |
    \d+
    |
    [^\w\s]
"""

custom_tokenizer = RegexpTokenizer(TOKEN_PATTERN, flags=re.VERBOSE)


def my_tokenize(text):
    return custom_tokenizer.tokenize(str(text))

In [24]:
print("========== TEST_LINES ==========")

for line in TEST_LINES:
    print("\nText:", line)
    print("Custom:", my_tokenize(line))
    print("NLTK:  ", word_tokenize(line))

print("\n========== GENERATED TICKETS ==========")

for ticket_id, text in zip(df["Ticket_ID"], raw_tickets):
    print(f"\n{ticket_id}: {text}")
    print("Custom:", my_tokenize(text))
    print("NLTK:  ", word_tokenize(text))

========== TEST_LINES ==========

Text: Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.
Custom: ['Mail', 'it.helpdesk@thapar.edu', 'or', 'a.b-c@dept.univ.ac.in', ';', 'visit', 'https://lms.thapar.edu/help', 'or', 'www.thapar.edu/it.']
NLTK:   ['Mail', 'it.helpdesk', '@', 'thapar.edu', 'or', 'a.b-c', '@', 'dept.univ.ac.in', ';', 'visit', 'https', ':', '//lms.thapar.edu/help', 'or', 'www.thapar.edu/it', '.']

Text: Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.
Custom: ['Call', '+91 98765 43210', ',', '+91-98765-43210', 'or', '0175-239-3201', '.', 'Version', '2.0.1', 'is', '95.5', '%', 'done', ';', 'fee', 'Rs', '1,250', '.']
NLTK:   ['Call', '+91', '98765', '43210', ',', '+91-98765-43210', 'or', '0175-239-3201', '.', 'Version', '2.0.1', 'is', '95.5', '%', 'done', ';', 'fee', 'Rs', '1,250', '.']

Text: The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.
Custo

In [25]:
different_tickets = []

for ticket_id, text in zip(df["Ticket_ID"], raw_tickets):
    custom = my_tokenize(text)
    nltk_tokens = word_tokenize(text)

    if custom != nltk_tokens:
        different_tickets.append(ticket_id)

print("Total tickets:", len(df))
print("Tickets tokenized differently:", len(different_tickets))
print("Ticket IDs:", different_tickets)

Total tickets: 14
Tickets tokenized differently: 4
Ticket IDs: ['T03', 'T04', 'T08', 'T10']


**Observe:** How many of your tickets are tokenized differently by my_tokenize and word_tokenize?

*Your answer:The custom tokenizer preserves contractions, hyphenated words, versions, decimals, emails, URLs and phone numbers as single tokens where their patterns match. NLTK's standard tokenizer uses different tokenization rules, so these formats can be split differently.*

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [ ]:
# Your code

**Observe:** Give one over-stemming and one under-stemming example from your table.

*Your answer:*

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [ ]:
# Your code

**Observe:** Why does TTR change from S1 to S4 in your data?

*Your answer:*